# TP1 — Linear Regression (California Housing)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session1/tp1.ipynb)

## Objective

**Regression** is a supervised learning task where we predict a **continuous value** (a number).

In this TP, we will predict the **house value** in California districts from features such as median income, average number of rooms, location, etc.

### Roadmap

1. **Explore** the data — understand what we're working with
2. **Visualize** — see relationships between features and the target
3. **Model** — train a linear regression
4. **Evaluate** — measure performance with proper methodology
5. **Improve** — feature scaling and cross-validation

> **Link to TP0:** In TP0, you learned how gradient descent finds optimal parameters by minimizing a loss function. Now we apply this idea to a real dataset using scikit-learn.

In [ ]:
!pip install scikit-learn

---
## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## 2. Load & Discover the Data

Before modeling, we must **understand our data**. What are the features? What are we predicting? What does the data look like?

This step is essential — you cannot build a good model on data you don't understand.

### Exercise 1.1 — Load the dataset

Load the California Housing dataset and display its shape and first rows.

**Why:** Understand the dimensions (how many samples? how many features?) and the content of our data.

*Hint:* `fetch_california_housing(as_frame=True)`, `df.head()`, `df.shape`

In [ ]:
# YOUR CODE HERE


These are features from the **California Housing dataset**:

- **MedInc** – Median income of households in the block group
- **HouseAge** – Median age of houses in the block group
- **AveRooms** – Average number of rooms per household
- **AveBedrms** – Average number of bedrooms per household
- **Population** – Total population in the block group
- **AveOccup** – Average number of occupants per household
- **Latitude** / **Longitude** – Geographic coordinates of the block group

**Target: MedHouseVal** – Median house value for the block group (in $100,000s).

### Exercise 1.2 — Summary statistics

Display summary statistics for each column.

**Why:** Understand the range and distribution of each feature — are they on the same scale? Are there extreme values?

*Hint:* `df.describe()`

**Question:** Which features have very different scales? Why could this be a problem for some models?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

### Exercise 1.3 — Check for missing values

Check if there are any missing values in the dataset.

**Why:** Missing data can break our models or bias results. It's always one of the first things to check.

*Hint:* `df.isnull().sum()`

In [ ]:
# YOUR CODE HERE


---
## 3. Data Visualization

Visualization helps us understand relationships in the data **before** modeling. It guides our intuition about which features might be useful predictors.

A good data scientist always visualizes before modeling.

### Exercise 2.1 — Distribution of the target variable

Plot the distribution of `MedHouseVal` (median house value).

**Why:** Understand what we're trying to predict — is the distribution symmetric? Are there outliers? Is the range reasonable?

*Hint:* `plt.hist()` or `sns.histplot()`

In [ ]:
# YOUR CODE HERE


### Exercise 2.2 — Scatter plots: features vs target

Create scatter plots of 2–3 features against the target variable.

**Why:** Visually identify which features correlate with house price. If we see a trend, that feature will be useful for prediction.

*Hint:* `plt.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.3)`

Suggested features: `MedInc` (median income) and `AveRooms` (average rooms).

In [ ]:
# YOUR CODE HERE


### Exercise 2.3 — Correlation heatmap

Compute and display the correlation matrix as a heatmap.

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

$$r = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y}$$

**Why:** Quantify the linear relationships between all features and the target. Correlation ranges from -1 (perfect negative) to +1 (perfect positive), with 0 meaning no linear relationship.

*Hint:* `sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')`

**Question:** Which feature has the strongest correlation with house price?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 4. Our First Model

### Linear Regression

Linear regression models the relationship between features and target as:

$$\hat{y} = X\theta + b$$

where $\theta$ are the model coefficients (one per feature) and $b$ is the intercept.

The model finds $\theta$ that minimizes the **loss function** — the Mean Squared Error (MSE):

$$\mathcal{L}(\theta) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

> **Link to TP0:** In TP0, you saw how gradient descent iteratively minimizes a loss function. scikit-learn's `LinearRegression` uses a closed-form solution (the Normal Equation) which computes the optimal $\theta$ directly — faster but mathematically equivalent.

### Exercise 3.1 — Separate features and target

Create the feature matrix `X` and the target vector `y`.

*Hint:* `X = df.drop('MedHouseVal', axis=1)`, `y = df['MedHouseVal']`

In [ ]:
# YOUR CODE HERE


### Exercise 3.2 — Train on all the data

Train a `LinearRegression` model on **all** the data and compute predictions.

**Why:** First, let's see the model in action before worrying about proper evaluation methodology.

*Hint:* `model = LinearRegression()`, `model.fit(X, y)`, `y_pred = model.predict(X)`

In [ ]:
# YOUR CODE HERE


### Exercise 3.3 — Predicted vs actual plot

Plot predicted values vs actual values (scatter plot), and add the perfect prediction line $y = x$.

**Why:** Visually assess how well the model fits. Points close to the diagonal line = good predictions.

*Hint:* `plt.scatter(y, y_pred, alpha=0.3)` and `plt.plot([min, max], [min, max], 'r--')`

In [ ]:
# YOUR CODE HERE


---
## 5. Train/Test Split — Why We Need It

### The Problem

In Exercise 3, we trained **and** evaluated on the **same** data. This is like studying the exam answers and then taking the same exam — of course you'll score well! But it doesn't mean you've learned.

A model can **memorize** training data without truly learning patterns (this is called **overfitting**). To honestly assess performance, we need data the model has **never seen**.

### The Solution: Train/Test Split

```
Full Dataset (20,640 samples)
├── Training Set (80%) → model learns from this
└── Test Set (20%)     → we evaluate on this (never seen during training)
```

The test set acts as a **fair exam** — the model has never seen these examples.

### Exercise 4.1 — Split the data

Split the data into training (80%) and test (20%) sets.

*Hint:* `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)`

In [ ]:
# YOUR CODE HERE


### Exercise 4.2 — Train and predict on both sets

Train the model on the **training set only**, then make predictions on both training and test sets.

**Why:** By comparing performance on train vs test, we can detect overfitting. If train performance is much better than test → the model has memorized rather than learned.

*Hint:* `model.fit(X_train, y_train)`, then compute predictions on both sets.

In [ ]:
# YOUR CODE HERE


---
## 6. Evaluation Metrics

How do we measure how good our predictions are? We need **metrics** — numbers that quantify prediction quality.

### Three Key Metrics for Regression

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** (Mean Squared Error) | $\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$ | Penalizes large errors more (squared). Same unit as $y^2$. |
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum_{i=1}^{n}\|y_i - \hat{y}_i\|$ | Average absolute mistake. Same unit as $y$. More interpretable. |
| **R²** (Coefficient of Determination) | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Proportion of variance explained. 1 = perfect, 0 = predicts mean. |

- **MSE** is the loss function we minimize during training
- **MAE** is easier to interpret ("on average, we're off by X")
- **R²** is scale-independent — useful for comparing across datasets

### Exercise 5.1 — Compute metrics on train and test sets

Compute MSE, MAE, and R² on both the training and test sets.

**Why:** Quantify model performance and detect overfitting by comparing train vs test scores.

*Hint:* `mean_squared_error(y_true, y_pred)`, `mean_absolute_error(...)`, `r2_score(...)`

**Question:** Is there a gap between train and test performance? What does it mean?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

### Exercise 5.2 — Predicted vs actual plot (test set)

Plot predicted vs actual values for the **test set**, and display R² on the plot.

**Why:** Combine visual and numeric assessment on unseen data.

In [ ]:
# YOUR CODE HERE


---
## 7. Feature Scaling

### Why Scale Features?

Our features have very different scales:
- `MedInc` ranges from 0 to ~15
- `Population` ranges from 3 to ~35,000
- `AveRooms` ranges from 1 to ~140

**`StandardScaler`** transforms each feature to have mean = 0 and standard deviation = 1:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

For **linear regression**, scaling doesn't change the predictions (the model compensates via coefficients), but it makes the coefficients **comparable** — you can directly read which feature matters most.

For many other algorithms (SVM, KNN, neural networks), scaling is **essential** for good performance.

### Exercise 6.1 — Scale features and retrain

Scale the features using `StandardScaler`, retrain the model, and compare metrics.

*Hint:* `scaler = StandardScaler()`, `X_train_scaled = scaler.fit_transform(X_train)`, `X_test_scaled = scaler.transform(X_test)`

**Important:** `fit_transform` on train set (learns mean/std), then only `transform` on test set (uses the same mean/std). Never fit on test data!

In [ ]:
# YOUR CODE HERE


### Exercise 6.2 — Compare coefficients

Compare model coefficients before and after scaling.

**Why:** After scaling, all features are on the same scale, so the coefficient magnitudes directly reflect each feature's importance.

*Hint:* `pd.Series(model.coef_, index=X.columns).sort_values()`

**Question:** Which feature contributes most to house price prediction?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 8. Cross-Validation

### The Problem with a Single Split

Our train/test split depends on `random_state` — a different split might give different results. How do we know our estimate is reliable?

### The Solution: K-Fold Cross-Validation

Cross-validation rotates the test set across **K folds** of the data:

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

Each sample is used as test exactly once. We report **mean ± standard deviation** of the metric → a more robust performance estimate.

### Exercise 7.1 — 5-fold cross-validation

Run 5-fold cross-validation and report R² mean ± std.

*Hint:* `scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')`

**Question:** Is the standard deviation large or small? What does that tell you about model stability?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 9. Summary & Key Takeaways

### What We Learned

1. **Explore** the data before modeling (shape, statistics, missing values)
2. **Visualize** to understand relationships (distributions, scatter plots, correlations)
3. **Split** data into train/test to honestly evaluate performance
4. **Train** a linear regression model
5. **Evaluate** with appropriate metrics (MSE, MAE, R²)
6. **Scale** features for interpretability and for other algorithms
7. **Cross-validate** for a robust performance estimate

### Key Formulas

| | Formula |
|---|---|
| **Linear model** | $\hat{y} = X\theta + b$ |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ |
| **Scaling** | $x_{\text{scaled}} = \frac{x - \mu}{\sigma}$ |

### Key Principle

**Always evaluate on unseen data.** A model that scores well on training data but poorly on test data has memorized rather than learned.